# 08 — Gold-Schicht und Datenqualität

## Zweck
Aus den Silver-Daten **analysebereite Gold-Tabellen** bauen und einen kurzen Qualitätsbericht
erstellen. Notebook `09` liest danach ausschließlich diese Gold-Tabellen — keine Transformationslogik mehr.

## Methodische Trennung
Historische **EEA-Messdaten** und der **Open-Meteo-Live-Snapshot** bleiben getrennt. Jede Tabelle trägt
`dataset_context` (`eea_historical` oder `open_meteo_live`). Sie dürfen nicht als identisch gelesen werden
(Messstation vs. Modellwert). Ein **explorativer** Vergleich (Gold 5) stellt den Live-Wert dennoch der
historischen Median-Baseline gegenüber — bewusst nur zur Lage-Einordnung, mit klarer Quellen-Kennzeichnung.

## Ausgabe (6 Parquet-Dateien in `data/gold/`)
`city_air_quality_daily_summary`, `pollutant_ranking_by_city`, `city_context_air_quality`,
`live_air_quality_latest`, `live_vs_historical_median`, `data_quality_summary`.

## Konfiguration und Silver-Daten laden

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

city_reference = pd.read_parquet(SILVER_DIR / "city_reference.parquet")
city_metadata = pd.read_parquet(SILVER_DIR / "city_metadata.parquet")
eea_daily = pd.read_parquet(SILVER_DIR / "eea_city_daily.parquet")
live_silver = pd.read_parquet(SILVER_DIR / "open_meteo_city_hourly")
print({"eea_daily": len(eea_daily), "live_silver": len(live_silver)})

{'eea_daily': 7938, 'live_silver': 192}


## Gold 1: Tageswerte je Stadt (eea_historical)
Die Silver-Tageswerte werden um den Stadtnamen ergänzt und einheitlich benannt.

In [2]:
daily_summary = (
    eea_daily.merge(city_reference[["city_id", "city_name"]], on="city_id", how="left")
    .rename(columns={"mean_value": "avg_value", "observation_count": "measurement_count"})
    .assign(dataset_context="eea_historical")
    [["city_id", "city_name", "date", "pollutant", "avg_value", "min_value", "max_value",
      "measurement_count", "source", "data_status", "dataset_context"]]
)
daily_summary.to_parquet(GOLD_DIR / "city_air_quality_daily_summary.parquet", index=False)
daily_summary.head()

,city_id,city_name,date,pollutant,avg_value,min_value,max_value,measurement_count,source,data_status,dataset_context
0,amsterdam_nl,Amsterdam,2025-01-01,no2,9.484470,2.1,41.3,264,eea_downloads_api,real_eea_api_postgres,eea_historical
1,amsterdam_nl,Amsterdam,2025-01-01,pm10,16.122876,0.8,227.3,153,eea_downloads_api,real_eea_api_postgres,eea_historical
2,amsterdam_nl,Amsterdam,2025-01-01,pm2_5,13.634167,0.7,205.5,120,eea_downloads_api,real_eea_api_postgres,eea_historical
3,amsterdam_nl,Amsterdam,2025-01-02,no2,27.071212,2.8,84.0,264,eea_downloads_api,real_eea_api_postgres,eea_historical
4,amsterdam_nl,Amsterdam,2025-01-02,pm10,10.191195,2.4,18.5,159,eea_downloads_api,real_eea_api_postgres,eea_historical


## Gold 2: Schadstoff-Rangfolge je Stadt (eea_historical)
Pro Schadstoff der Stadtmittelwert über den Zeitraum, absteigend gereiht.

In [3]:
ranking = (
    daily_summary.groupby(["pollutant", "city_id", "city_name"], as_index=False)
    .agg(avg_value=("avg_value", "mean"),
         days=("date", "nunique"),
         measurement_count=("measurement_count", "sum"))
)
ranking["rank"] = ranking.groupby("pollutant")["avg_value"].rank(ascending=False, method="min").astype(int)
ranking["dataset_context"] = "eea_historical"
ranking = ranking.sort_values(["pollutant", "rank"])
ranking.to_parquet(GOLD_DIR / "pollutant_ranking_by_city.parquet", index=False)
ranking.head(10)

,pollutant,city_id,city_name,avg_value,days,measurement_count,rank,dataset_context
5,no2,rome_it,Rome,25.009749,364,106642,1,eea_historical
3,no2,paris_fr,Paris,22.968982,365,266390,2,eea_historical
4,no2,prague_cz,Prague,22.663012,365,74462,3,eea_historical
7,no2,warsaw_pl,Warsaw,22.553729,365,33928,4,eea_historical
2,no2,madrid_es,Madrid,22.481703,365,281374,5,eea_historical
0,no2,amsterdam_nl,Amsterdam,18.100652,353,85955,6,eea_historical
6,no2,vienna_at,Vienna,16.030066,348,125040,7,eea_historical
1,no2,berlin_de,Berlin,15.331012,365,122310,8,eea_historical
14,pm10,warsaw_pl,Warsaw,21.041079,365,51449,1,eea_historical
12,pm10,prague_cz,Prague,20.736136,365,104348,2,eea_historical


## Gold 3: Rangfolge mit Stadtkontext (eea_historical)
Die Rangfolge wird mit den Wikipedia-Metadaten (Bevölkerungsdichte) verknüpft — als **explorativer**
Kontext, nicht als Erklärung.

In [4]:
context = ranking.merge(
    city_metadata[["city_id", "population", "area_km2", "population_density",
                   "density_comparable", "area_basis_note"]],
    on="city_id", how="left",
)
context.to_parquet(GOLD_DIR / "city_context_air_quality.parquet", index=False)
context.head(10)

,pollutant,city_id,city_name,avg_value,days,measurement_count,rank,dataset_context,population,area_km2,population_density,density_comparable,area_basis_note
0,no2,rome_it,Rome,25.009749,364,106642,1,eea_historical,2746984,1287.36,2133.81,True,
1,no2,paris_fr,Paris,22.968982,365,266390,2,eea_historical,2047602,105.40,19430.00,False,"Kernkommune (20 Arrondissements, ~105 km²) — n..."
2,no2,prague_cz,Prague,22.663012,365,74462,3,eea_historical,1407084,496.21,2835.70,True,
3,no2,warsaw_pl,Warsaw,22.553729,365,33928,4,eea_historical,1862402,517.24,3500.00,True,
4,no2,madrid_es,Madrid,22.481703,365,281374,5,eea_historical,3477497,605.77,5740.60,True,
5,no2,amsterdam_nl,Amsterdam,18.100652,353,85955,6,eea_historical,933680,219.32,5277.00,True,
6,no2,vienna_at,Vienna,16.030066,348,125040,7,eea_historical,2028499,414.78,4890.50,True,
7,no2,berlin_de,Berlin,15.331012,365,122310,8,eea_historical,3596999,891.30,4109.00,True,
8,pm10,warsaw_pl,Warsaw,21.041079,365,51449,1,eea_historical,1862402,517.24,3500.00,True,
9,pm10,prague_cz,Prague,20.736136,365,104348,2,eea_historical,1407084,496.21,2835.70,True,


## Gold 4: Live-Snapshot je Stadt (open_meteo_live)
Aus dem Spark-Streaming-Output je Stadt der **neueste** Messzeitpunkt.

In [5]:
live_silver["event_time_ts"] = pd.to_datetime(live_silver["event_time_ts"], utc=True)
latest_idx = live_silver.groupby("city_id")["event_time_ts"].idxmax()
live_latest = (
    live_silver.loc[latest_idx]
    .assign(dataset_context="open_meteo_live")
    [["city_id", "city_name", "event_time_ts", "pm2_5", "pm10", "no2", "dataset_context"]]
    .reset_index(drop=True)
)
live_latest.to_parquet(GOLD_DIR / "live_air_quality_latest.parquet", index=False)
live_latest

,city_id,city_name,event_time_ts,pm2_5,pm10,no2,dataset_context
0,amsterdam_nl,Amsterdam,2026-07-01 23:00:00+00:00,15.7,22.3,39.5,open_meteo_live
1,berlin_de,Berlin,2026-07-01 23:00:00+00:00,5.5,9.9,19.0,open_meteo_live
2,madrid_es,Madrid,2026-07-01 23:00:00+00:00,2.7,4.7,3.2,open_meteo_live
3,paris_fr,Paris,2026-07-01 23:00:00+00:00,3.6,6.4,10.9,open_meteo_live
4,prague_cz,Prague,2026-07-01 23:00:00+00:00,4.4,6.1,5.9,open_meteo_live
5,rome_it,Rome,2026-07-01 23:00:00+00:00,10.5,19.3,8.0,open_meteo_live
6,vienna_at,Vienna,2026-07-01 23:00:00+00:00,3.1,3.6,4.3,open_meteo_live
7,warsaw_pl,Warsaw,2026-07-01 23:00:00+00:00,10.1,12.3,7.7,open_meteo_live


## Gold 5: Live-Wert vs. historischer EEA-Median 2025 (explorativ)
Eine **explorative** Einordnung der aktuellen Lage: der Open-Meteo-Modellwert je Stadt gegen den
**Median** der gemessenen EEA-Tageswerte 2025 (mit 25%-/75%-Quartilen als Streubreite). Bewusst
quellenübergreifend — Modell-Snapshot gegen Mess-Baseline —, daher nur zur Lage-Einordnung:
**keine** Gleichsetzung der Quellen, keine Kausalität, kein WHO-Jahresvergleich. `delta_abs` und
`delta_pct` zeigen die Abweichung vom Median. **Rom** behält seine Live-Werte; wo keine validierte
historische PM-Referenz existiert, bleibt der Vergleich offen (`comparison_status = no_historical_reference`).

In [6]:
# Pro Stadt und Schadstoff: Median, Quartile und Anzahl der EEA-Tageswerte 2025 (Baseline).
historical_median = (
    daily_summary
    .groupby(["city_id", "city_name", "pollutant"], as_index=False)
    .agg(
        historical_median_value=("avg_value", "median"),
        historical_q25=("avg_value", lambda s: s.quantile(0.25)),
        historical_q75=("avg_value", lambda s: s.quantile(0.75)),
        historical_days=("avg_value", "count"),
    )
)

# Live-Snapshot vom Wide- ins Long-Format bringen (eine Zeile je Stadt und Schadstoff).
live_long = live_latest.melt(
    id_vars=["city_id", "city_name", "event_time_ts", "dataset_context"],
    value_vars=["pm2_5", "pm10", "no2"],
    var_name="pollutant",
    value_name="live_value",
)

# Live gegen historische Baseline; fehlt der Median (z. B. Rom PM), bleibt der Vergleich offen.
live_vs_historical_median = live_long.merge(
    historical_median, on=["city_id", "city_name", "pollutant"], how="left"
)
live_vs_historical_median["comparison_status"] = (
    live_vs_historical_median["historical_median_value"].notna()
    .map({True: "comparable", False: "no_historical_reference"})
)
live_vs_historical_median["delta_abs"] = (
    live_vs_historical_median["live_value"] - live_vs_historical_median["historical_median_value"]
)
live_vs_historical_median["delta_pct"] = (
    live_vs_historical_median["delta_abs"] / live_vs_historical_median["historical_median_value"] * 100
)
live_vs_historical_median["comparison_note"] = (
    "Explorative Einordnung: Open-Meteo-Modellwert als aktueller Snapshot gegen historischen "
    "EEA-Median 2025; Quellen nicht als identisch interpretieren."
)

live_vs_historical_median = live_vs_historical_median[[
    "city_id", "city_name", "pollutant", "event_time_ts",
    "live_value", "historical_median_value", "historical_q25", "historical_q75", "historical_days",
    "delta_abs", "delta_pct", "comparison_status", "comparison_note",
]]

live_vs_historical_median.to_parquet(GOLD_DIR / "live_vs_historical_median.parquet", index=False)
live_vs_historical_median.sort_values(["pollutant", "city_name"])

,city_id,city_name,pollutant,event_time_ts,live_value,historical_median_value,historical_q25,historical_q75,historical_days,delta_abs,delta_pct,comparison_status,comparison_note
16,amsterdam_nl,Amsterdam,no2,2026-07-01 23:00:00+00:00,39.5,16.152308,12.214773,22.183712,353.0,23.347692,144.547100,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
17,berlin_de,Berlin,no2,2026-07-01 23:00:00+00:00,19.0,14.335923,10.377232,18.531048,365.0,4.664077,32.534198,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
18,madrid_es,Madrid,no2,2026-07-01 23:00:00+00:00,3.2,19.601573,14.089286,29.167098,365.0,-16.401573,-83.674779,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
19,paris_fr,Paris,no2,2026-07-01 23:00:00+00:00,10.9,21.465541,15.752789,28.646056,365.0,-10.565541,-49.220939,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
20,prague_cz,Prague,no2,2026-07-01 23:00:00+00:00,5.9,21.270531,16.193171,26.666990,365.0,-15.370531,-72.262094,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
21,rome_it,Rome,no2,2026-07-01 23:00:00+00:00,8.0,23.593644,18.911818,31.369808,364.0,-15.593644,-66.092563,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
22,vienna_at,Vienna,no2,2026-07-01 23:00:00+00:00,4.3,14.414658,10.739490,20.125092,348.0,-10.114658,-70.169254,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
23,warsaw_pl,Warsaw,no2,2026-07-01 23:00:00+00:00,7.7,21.090625,15.811458,27.052083,365.0,-13.390625,-63.490888,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
8,amsterdam_nl,Amsterdam,pm10,2026-07-01 23:00:00+00:00,22.3,14.870833,11.811310,19.577459,351.0,7.429167,49.957971,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...
9,berlin_de,Berlin,pm10,2026-07-01 23:00:00+00:00,9.9,15.374242,11.947955,20.445455,365.0,-5.474242,-35.606583,comparable,Explorative Einordnung: Open-Meteo-Modellwert ...


In [7]:
assert len(live_vs_historical_median) == 24, "Erwartet: 8 Städte × 3 Schadstoffe = 24 Zeilen."
assert live_vs_historical_median["city_id"].nunique() == 8, "Es müssen 8 Städte enthalten sein."
assert set(live_vs_historical_median["pollutant"]) == {"pm2_5", "pm10", "no2"}, \
    "Es müssen genau pm2_5, pm10 und no2 enthalten sein."

status_counts = live_vs_historical_median["comparison_status"].value_counts().to_dict()
assert status_counts.get("comparable", 0) == 22, \
    f"Erwartet: 22 vergleichbare Zeilen, erhalten: {status_counts}"
assert status_counts.get("no_historical_reference", 0) == 2, \
    f"Erwartet: 2 Zeilen ohne historische Referenz, erhalten: {status_counts}"

rome_pm_missing = live_vs_historical_median[
    (live_vs_historical_median["city_id"] == "rome_it")
    & (live_vs_historical_median["pollutant"].isin(["pm2_5", "pm10"]))
]
assert len(rome_pm_missing) == 2, "Rom muss für pm2_5 und pm10 ohne historische Referenz enthalten sein."
assert rome_pm_missing["comparison_status"].eq("no_historical_reference").all(), \
    "Rom pm2_5/pm10 müssen als no_historical_reference markiert sein."

comparable = live_vs_historical_median[
    live_vs_historical_median["comparison_status"] == "comparable"
]
assert comparable["historical_median_value"].notna().all(), \
    "Alle vergleichbaren Zeilen brauchen einen historischen Median."
assert comparable["delta_abs"].notna().all(), "Alle vergleichbaren Zeilen brauchen delta_abs."
assert comparable["delta_pct"].notna().all(), "Alle vergleichbaren Zeilen brauchen delta_pct."

print("OK: live_vs_historical_median erfüllt die erwartete Struktur.")

OK: live_vs_historical_median erfüllt die erwartete Struktur.


## Gold 6: Qualitätsbericht
Zeilenzahlen, fehlende Werte und der je Tabelle abgedeckte Zeitraum (`coverage_days`) — die Grundlage,
auf der Notebook `09` seine Aussagen einordnet. Für die EEA-historischen Tabellen ist das die Zahl der
abgedeckten Tage; für den Open-Meteo-Live-Snapshot und den quellenübergreifenden Vergleich ist sie
**nicht zutreffend** (`NA`). Dass `live_vs_historical_median` fehlende Werte ausweist, ist **korrekt**:
Rom hat für PM2.5/PM10 keine historische Referenz.

In [8]:
gold_tables = {
    "city_air_quality_daily_summary": daily_summary,
    "pollutant_ranking_by_city": ranking,
    "city_context_air_quality": context,
    "live_air_quality_latest": live_latest,
    "live_vs_historical_median": live_vs_historical_median,
}
HISTORICAL_DAYS = int(daily_summary["date"].nunique())


def coverage_days(df: pd.DataFrame):
    # Tabellen ohne einheitlichen dataset_context (z. B. der quellenübergreifende Vergleich)
    # oder der reine Open-Meteo-Live-Snapshot decken keinen historischen Zeitraum ab -> NA.
    if "dataset_context" not in df.columns:
        return pd.NA
    if set(df["dataset_context"].unique()) == {"open_meteo_live"}:
        return pd.NA
    return HISTORICAL_DAYS


quality_summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns),
     "missing_values": int(df.isna().sum().sum()),
     "coverage_days": coverage_days(df)}
    for name, df in gold_tables.items()
])
quality_summary["coverage_days"] = quality_summary["coverage_days"].astype("Int64")
quality_summary.to_parquet(GOLD_DIR / "data_quality_summary.parquet", index=False)

print({"historische_tage": HISTORICAL_DAYS})
quality_summary

{'historische_tage': 365}


,table,rows,columns,missing_values,coverage_days
0,city_air_quality_daily_summary,7938,11,0,365
1,pollutant_ranking_by_city,22,8,0,365
2,city_context_air_quality,22,13,0,365
3,live_air_quality_latest,8,7,0,<NA>
4,live_vs_historical_median,24,13,12,<NA>


## Nächster Schritt
Notebook `09` ausführen — Analyse, Visualisierung und Ergebnisgeschichte.